# SpamShield AI - Classic NLP Baseline

Traditional ML: TF-IDF + Logistic Regression, Naive Bayes, SVM, Random Forest.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q scikit-learn nltk matplotlib seaborn pandas

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, roc_auc_score
)
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

In [ ]:
DATA_PATH = '/content/drive/My Drive/data/sms_spam.parquet'

df = pd.read_parquet(DATA_PATH)
print(f'Total: {len(df):,}')
print(f'Ham: {(df["label"] == 0).sum():,} ({(df["label"] == 0).mean()*100:.1f}%)')
print(f'Spam: {(df["label"] == 1).sum():,} ({(df["label"] == 1).mean()*100:.1f}%)')
df.head()

## 2. Text Preprocessing

In [ ]:
STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

df['cleaned'] = df['text'].apply(clean_text)

print('Sample cleaned text:')
print(f'  Original: {df.iloc[0]["text"][:100]}')
print(f'  Cleaned:  {df.iloc[0]["cleaned"][:100]}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned'].values, df['label'].values,
    test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

## 3. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'TF-IDF shape: {X_train_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,}')

In [ ]:
feature_names = tfidf.get_feature_names_out()

for label, name in [(0, 'Ham'), (1, 'Spam')]:
    mask = y_train == label
    mean_tfidf = X_train_tfidf[mask].mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-15:][::-1]
    top_words = [(feature_names[i], mean_tfidf[i]) for i in top_idx]
    print(f'\nTop {name} words:')
    for word, score in top_words:
        print(f'  {word:<20} {score:.4f}')

## 4. Train Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'SVM': LinearSVC(max_iter=2000, C=1.0, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results[name] = {
        'model': model,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'y_pred': y_pred
    }
    print(f'  Acc: {acc:.4f} | F1: {f1:.4f}')

## 5. Compare Models

In [ ]:
results_df = pd.DataFrame({
    name: {k: v for k, v in r.items() if k not in ['model', 'y_pred']}
    for name, r in results.items()
}).T

print(results_df.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
results_df[['accuracy', 'precision', 'recall', 'f1']].plot(kind='bar', ax=ax)
ax.set_title('Model Comparison')
ax.set_ylabel('Score')
ax.set_ylim(0.9, 1.0)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
best_name = results_df['f1'].idxmax()
print(f'Best model: {best_name} (F1: {results_df.loc[best_name, "f1"]:.4f})')
print(classification_report(y_test, results[best_name]['y_pred'], target_names=['Ham', 'Spam']))

## 6. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 5))

for idx, (name, r) in enumerate(results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
    axes[idx].set_title(f'{name}\nF1: {r["f1"]:.4f}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 7. Error Analysis

In [ ]:
best_pred = results[best_name]['y_pred']

fp_idx = [i for i in range(len(y_test)) if y_test[i] == 0 and best_pred[i] == 1]
fn_idx = [i for i in range(len(y_test)) if y_test[i] == 1 and best_pred[i] == 0]

print(f'Best model: {best_name}')
print(f'False Positives: {len(fp_idx)}')
print(f'False Negatives: {len(fn_idx)}')

if fp_idx:
    print(f'\nSample False Positives (Ham -> Spam):')
    for i in fp_idx[:5]:
        print(f'  {X_test[i][:80]}')

if fn_idx:
    print(f'\nSample False Negatives (Spam -> Ham):')
    for i in fn_idx[:5]:
        print(f'  {X_test[i][:80]}')

## 8. Custom Predictions

In [ ]:
best_model = results[best_name]['model']

def predict(text):
    cleaned = clean_text(text)
    vectorized = tfidf.transform([cleaned])
    pred = best_model.predict(vectorized)[0]
    return 'Spam' if pred == 1 else 'Ham'

test_messages = [
    "Hey, are we still meeting for lunch tomorrow?",
    "Congratulations! You've won a $1000 gift card. Click here to claim now!",
    "URGENT: Your account has been compromised. Verify your password immediately.",
    "Meeting moved to 3pm. See you there.",
    "FREE FREE FREE! Win a car by texting WIN to 80085",
    "Can you pick up some milk on your way home?",
    "You have been selected for a cash prize! Call now!",
    "Thanks for dinner last night, it was great!"
]

print(f'Predictions ({best_name}):')
print('-' * 80)
for msg in test_messages:
    label = predict(msg)
    print(f'[{label:<4}] {msg[:70]}')

## 9. Summary

In [ ]:
print('=' * 60)
print('CLASSIC NLP BASELINE RESULTS')
print('=' * 60)
print(f'\nDataset: SMS Spam Collection ({len(df):,} messages)')
print(f'Features: TF-IDF (5k features, unigrams + bigrams)')
print(f'\nResults:')
for name, r in sorted(results.items(), key=lambda x: x[1]['f1'], reverse=True):
    print(f'  {name:<25} Acc: {r["accuracy"]:.4f}  F1: {r["f1"]:.4f}')
print(f'\nBest: {best_name} (F1: {results[best_name]["f1"]:.4f})')